Import

In [2]:
import pandas as pd
import os

Load Datasheets

In [5]:
movies_path = "D:/Entertainment_AI_Project/Data/Raw/movies.csv"
ratings_path = "D:/Entertainment_AI_Project/Data/Raw/ratings.csv"

print("Loading datasets...")
movies = pd.read_csv(movies_path)
ratings = pd.read_csv(ratings_path)

Loading datasets...


Basic Inspection

In [7]:
print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)

print("\nMovies columns:", movies.columns.tolist())
print("Ratings columns:", ratings.columns.tolist())

print("\nMissing values in movies:")
print(movies.isnull().sum())

print("\nMissing values in ratings:")
print(ratings.isnull().sum())

print("\nDuplicate rows in movies:", movies.duplicated().sum())
print("Duplicate rows in ratings:", ratings.duplicated().sum())

Movies shape: (62423, 3)
Ratings shape: (25000095, 4)

Movies columns: ['movieId', 'title', 'genres']
Ratings columns: ['userId', 'movieId', 'rating', 'timestamp']

Missing values in movies:
movieId    0
title      0
genres     0
dtype: int64

Missing values in ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Duplicate rows in movies: 0
Duplicate rows in ratings: 0


Cleaning 

In [8]:
movies.drop_duplicates(inplace=True)
ratings.drop_duplicates(inplace=True)

movies.dropna(subset=["movieId", "title", "genres"], inplace=True)
ratings.dropna(subset=["userId", "movieId", "rating", "timestamp"], inplace=True)

movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")

movies["clean_title"] = movies["title"].str.replace(r"\(\d{4}\)", "", regex=True).str.strip()

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s", errors="coerce")
ratings = ratings[(ratings["rating"] >= 0.5) & (ratings["rating"] <= 5.0)]

Merge

In [9]:
movie_data = pd.merge(ratings, movies, on="movieId", how="left")
print("Merged dataset shape:", movie_data.shape)
movie_data.head()

Merged dataset shape: (25000095, 8)


,userId,movieId,rating,timestamp,title,genres,year,clean_title
0,1,296,5.0,2006-05-17 15:34:04,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,1994.0,Pulp Fiction
1,1,306,3.5,2006-05-17 12:26:57,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,1994.0,Three Colors: Red (Trois couleurs: Rouge)
2,1,307,5.0,2006-05-17 12:27:08,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama,1993.0,Three Colors: Blue (Trois couleurs: Bleu)
3,1,665,5.0,2006-05-17 15:13:40,Underground (1995),Comedy|Drama|War,1995.0,Underground
4,1,899,3.5,2006-05-17 12:21:50,Singin' in the Rain (1952),Comedy|Musical|Romance,1952.0,Singin' in the Rain


Filter active users and popular movies

In [10]:
user_counts = movie_data["userId"].value_counts()
movie_counts = movie_data["movieId"].value_counts()

active_users = user_counts[user_counts >= 20].index
popular_movies = movie_counts[movie_counts >= 20].index

filtered_data = movie_data[
    (movie_data["userId"].isin(active_users)) &
    (movie_data["movieId"].isin(popular_movies))
]

print("Filtered dataset shape:", filtered_data.shape)
filtered_data.head()

Filtered dataset shape: (24810483, 8)


,userId,movieId,rating,timestamp,title,genres,year,clean_title
0,1,296,5.0,2006-05-17 15:34:04,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,1994.0,Pulp Fiction
1,1,306,3.5,2006-05-17 12:26:57,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,1994.0,Three Colors: Red (Trois couleurs: Rouge)
2,1,307,5.0,2006-05-17 12:27:08,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama,1993.0,Three Colors: Blue (Trois couleurs: Bleu)
3,1,665,5.0,2006-05-17 15:13:40,Underground (1995),Comedy|Drama|War,1995.0,Underground
4,1,899,3.5,2006-05-17 12:21:50,Singin' in the Rain (1952),Comedy|Musical|Romance,1952.0,Singin' in the Rain


Save Processed Files 

In [11]:
os.makedirs("data/processed", exist_ok=True)

movies.to_csv("data/processed/movies_clean.csv", index=False)
ratings.to_csv("data/processed/ratings_clean.csv", index=False)
movie_data.to_csv("data/processed/movie_data_merged.csv", index=False)
filtered_data.to_csv("data/processed/movie_data_filtered.csv", index=False)

print("Preprocessing complete.")

Preprocessing complete.
